# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanikawaghmode-mumbai/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

The purpose of this queue is to turn the observed signal analysis into practical, human-reviewed content actions.

The ranking is based on the measured signals from the validated analysis. The queue should be treated as a prioritisation tool rather than an automatic decision system.

### Reason codes

* **HIGH_SIGNAL:** the signal showed a relatively strong measured association with the ranking-related outcome.
* **STABLE_SIGNAL:** the relationship remained directionally similar in the validation check.
* **UNSTABLE_SIGNAL:** the relationship changed substantially between the original and grouped analysis and therefore requires additional review.
* **DATA_LIMIT:** the available data is insufficient to support a confident action.
* **REFRESH_REVIEW:** the observed pattern suggests that the content may be worth reviewing for freshness or update opportunities.

Each recommendation should include the signal, reason code, suggested action, and a human-review requirement.


In [ ]:
# Build a ranked action queue from the validated signal comparison

queue = comparison.copy().reset_index()

queue = queue.rename(columns={
    "index": "signal"
})

queue["abs_test_correlation"] = queue["grouped_test"].abs()

queue = queue.sort_values(
    "abs_test_correlation",
    ascending=False
)

queue["rank"] = range(1, len(queue) + 1)

def assign_reason(row):
    if pd.isna(row["grouped_test"]):
        return "DATA_LIMIT"

    if abs(row["train_test_difference"]) > 0.20:
        return "UNSTABLE_SIGNAL"

    if abs(row["grouped_test"]) >= 0.30:
        return "HIGH_SIGNAL"

    return "STABLE_SIGNAL"

queue["reason_code"] = queue.apply(assign_reason, axis=1)

queue["suggested_action"] = queue["reason_code"].map({
    "HIGH_SIGNAL": "Review relevant content and investigate whether a refresh or optimisation opportunity exists.",
    "STABLE_SIGNAL": "Review the associated content pattern and consider whether further investigation is useful.",
    "UNSTABLE_SIGNAL": "Validate the signal across additional clients or time periods before taking action.",
    "DATA_LIMIT": "Collect or verify additional data before making a content decision."
})

queue["human_review"] = "REQUIRED"

display(
    queue[
        [
            "rank",
            "signal",
            "reason_code",
            "suggested_action",
            "human_review"
        ]
    ].head(20)
)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The playbook is intended to help a content team prioritise which content signals and content groups deserve human investigation.

It is a decision-support tool for research and prioritisation. It is not intended to automatically change published content, rankings, metadata, or client strategies.

### Limits

The analysis is based on the available anonymised dataset and the validated signal relationships.

Observed associations do not establish causation. The results may also vary across clients, time periods, content types, and data availability.

The ranked queue should therefore be treated as directional evidence rather than a guaranteed prediction of ranking improvement.

The queue should be reviewed by a human before any content action is taken.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

### Human-review rules

Every recommendation in the queue requires human review before action.

The reviewer should:

1. Check the actual content behind the recommendation.
2. Confirm that the signal is relevant to that content.
3. Check whether the recommendation makes sense for the content's purpose and audience.
4. Check for missing or unusual data.
5. Confirm that the proposed change would not reduce useful information or content quality.
6. Record whether the recommendation was accepted, rejected, or sent for further investigation.

### What should NOT be automated

The system should not automatically:

* publish or edit content;
* delete content;
* change titles or URLs;
* change client strategy;
* make claims about ranking causation;
* guarantee ranking improvement;
* make decisions based only on a single signal;
* override editorial or subject-matter review;
* make irreversible content changes.

The output is a research and prioritisation queue, not a production automation system.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The analysis should be periodically rechecked to determine whether the observed relationships remain directionally similar.

### Monitoring checks

Monitor:

* changes in the distribution of the main signals;
* changes in the ranking-related outcome;
* changes in the measured correlation of important signals;
* missing-data rates;
* changes in the number of observations and clients;
* changes in the stability of the grouped validation results.

### Review triggers

A new validation review should be considered when:

* a major data-source change occurs;
* the definition of the outcome changes;
* important features are added or removed;
* the measured signal relationship changes substantially;
* missing data increases materially;
* the client or content population changes substantially.

These triggers are intended to prompt human review and re-analysis. They are not automatic production retraining rules.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os
import json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

print("Output folders ready.")

In [ ]:
# Export the ranked recommendation queue

output_columns = [
    "rank",
    "signal",
    "reason_code",
    "suggested_action",
    "human_review"
]

final_queue = queue[output_columns].copy()

output_path = "work/outputs/ranked_action_queue.csv"

final_queue.to_csv(
    output_path,
    index=False
)

print("Queue exported to:", output_path)

display(final_queue.head(20))

In [ ]:
metrics = {
    "analysis": "Ranking Signal Analysis",
    "method": "Spearman correlation",
    "validation": "client-grouped",
    "group_overlap": int(len(overlap)),
    "number_of_signals": int(len(feature_columns)),
    "number_of_ranked_actions": int(len(final_queue))
}

metrics_path = "work/outputs/action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved to:", metrics_path)

In [ ]:
import matplotlib.pyplot as plt

plot_df = comparison.copy()

plot_df = plot_df.sort_values(
    "grouped_test",
    key=abs,
    ascending=False
).head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_df.index.astype(str),
    plot_df["grouped_test"]
)

plt.xlabel("Spearman correlation in grouped test")
plt.ylabel("Signal")
plt.title("Top observed ranking-related signal associations")

plt.tight_layout()

figure_path = "work/figures/top_signal_associations.png"

plt.savefig(
    figure_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Figure saved to:", figure_path)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.